In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

reponse = llm.invoke("Dis bonjour en une phrase.")

print("RÉPONSE :", reponse.content)
print("RAISONNEMENT :", reponse.additional_kwargs.get("reasoning_content", "(aucun)")[:200])

RÉPONSE : Bonjour !
RAISONNEMENT : Okay, the user wants me to say "Dis bonjour en une phrase." which translates to "Say hello in one sentence." in French.

First, I need to make sure I understand the request correctly. They want a gree


In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte."
)

In [3]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Je m'appelle Houssam"}]
})
print(response["messages"][-1].content)

Bonjour Houssam ! 😊


In [4]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Quel est mon prénom ?"}]
})
print(response["messages"][-1].content)

Je ne sais pas votre prénom.


In [5]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

In [6]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Je m'appelle Houssam"}]},
    config=config
)
print(res["messages"][-1].content)

Bonjour Houssam ! 😊


In [7]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config
)
print(res["messages"][-1].content)

Houssam 😊


In [8]:
config2 = {"configurable": {"thread_id": "2"}}

res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config2
)
print(res["messages"][-1].content)

Je ne sais pas votre prénom.


## Étape 2 : Tools personnalisés

In [9]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Donne la météo actuelle d'une ville."""
    print(f"[TOOL] get_weather appelé pour {location}")
    return f"Météo à {location} : ensoleillé, 32°C"

@tool
def get_employee_info(name: str) -> dict:
    """Donne le salaire et le poste d'un employé à partir de son prénom."""
    print(f"[TOOL] get_employee_info appelé pour {name}")
    return {"name": name, "salary": 12000, "job": "Développeur"}

@tool
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres."""
    print(f"[TOOL] multiply appelé avec a={a}, b={b}")
    return a * b

tools = [get_weather, get_employee_info, multiply]

In [10]:
print(get_weather.invoke({"location": "Marrakech"}))
print(multiply.invoke({"a": 6, "b": 7}))
print(get_weather.name, "-", get_weather.description)

[TOOL] get_weather appelé pour Marrakech
Météo à Marrakech : ensoleillé, 32°C
[TOOL] multiply appelé avec a=6.0, b=7.0
42.0
get_weather - Donne la météo actuelle d'une ville.


In [11]:
from langgraph.checkpoint.memory import InMemorySaver

agent_tools = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Tu es un assistant utile. Utilise les outils quand c'est nécessaire. Réponds en français.",
    checkpointer=InMemorySaver()
)

config_tools = {"configurable": {"thread_id": "tools-1"}}

In [12]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Casablanca ?"}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_weather appelé pour Casablanca
La météo à Casablanca est actuellement ensoleillée à 32°C.


In [13]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quel est le salaire de Mohamed ? Multiplie-le par 3."}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_employee_info appelé pour Mohamed
[TOOL] multiply appelé avec a=12000.0, b=3.0
Le salaire de Mohamed est de 12 000. Multiplié par 3, cela donne 36 000.


## Étape 3 : Tools prédéfinis

In [18]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg = DuckDuckGoSearchRun()
print(ddg.invoke("Quelle est la capitale du Maroc ?")[:400])

TimeoutException: error decoding response body > request or response body error > operation timed out

> DuckDuckGo : requête expirée (timeout réseau). Voir Tavily ci-dessous.

In [17]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv(override=True)

tavily = TavilySearch(max_results=3)
resultat = tavily.invoke({"query": "Quelle est la capitale du Maroc ?"})

for r in resultat["results"]:
    print("-", r["title"])
    print("  ", r["url"])

- Quelle est la capitale du Maroc et pourquoi ce choix ?
   https://citopendia.fr/quelle-est-la-capitale-du-maroc-et-pourquoi-ce-choix
- Quelle est la capitale du Maroc ?
   https://darijaschool.com/quelle-est-la-capitale-du-maroc
- Quelle est la capitale du Népal ?
   https://memoriser-culture-generale.fr/quelle-est-la-capitale-du-nepal


In [19]:
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
print(python_repl.invoke("print(2 ** 10)"))

C:\Users\bouho\AppData\Local\Temp\ipykernel_25152\1561934165.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
Python REPL can execute arbitrary code. Use with caution.


1024



In [20]:
from langgraph.checkpoint.memory import InMemorySaver

tools_pre = [tavily, python_repl]

agent_pre = create_agent(
    model=llm,
    tools=tools_pre,
    system_prompt=(
        "Tu es un assistant utile. Utilise l'outil de recherche web pour les "
        "informations récentes, et l'outil Python pour les calculs. Réponds en français."
    ),
    checkpointer=InMemorySaver()
)

config_pre = {"configurable": {"thread_id": "pre-1"}}

In [22]:
def poser(question):
    res = agent_pre.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config_pre
    )
    for m in res["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("[APPEL OUTIL]", tc["name"], tc["args"])
    print("\nRÉPONSE :", res["messages"][-1].content)

In [23]:
poser("Cherche sur le web quelle est la capitale du Maroc et donne-moi une source.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}

RÉPONSE : La capitale du Maroc est **Rabat**.  
Source : [Wikipédia](https://fr.wikipedia.org/wiki/Maroc) (article "Maroc").  

*Explication : Selon la Wikipédia, "La capitale administrative et politique du Maroc est Rabat tandis que la capitale économique et la plus grande ville du pays est Casablanca".*


In [24]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.


In [25]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.
